# Task 5 — Spark metadata stream to MongoDB

Evidence run metadata is read from `screenshots/stage2_manifest.json`. Commands: `bash scripts/capture_spark_evidence.sh` and `bash scripts/capture_store_evidence.sh`. Spark consumes only `cpg.metadata`, checkpoints the Kafka offset, and upserts by `file_id`. Raw query output: [`metadata_evidence.txt`](../screenshots/mongodb/metadata_evidence.txt).


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'screenshots/stage2_manifest.json').exists())
manifest = json.loads((ROOT / 'screenshots/stage2_manifest.json').read_text())
mongo = manifest['metrics']['mongodb']; spark = manifest['metrics']['spark']
print('MongoDB documents:', mongo['documents'])
print('unique file IDs/paths:', mongo['unique_file_ids'], '/', mongo['unique_file_paths'])
print('duplicate file_id groups:', mongo['duplicate_file_id_groups'])
print('Spark metadata offset:', spark['metadata_offset'])
print('completed micro-batch:', spark['completed_batch'])
assert mongo['documents'] == spark['metadata_offset'] == 5


MongoDB documents: 5
unique file IDs/paths: 5 / 5
duplicate file_id groups: 0
Spark metadata offset: 5
completed micro-batch: 0


## Final database UI evidence

The image below is real Mongo Express evidence from the canonical Stage 3 replay final state, after metadata replacement. UI action/query: filter the target `file_id`. Run date: `2026-07-17`. Result: `pass`. The screenshot is labeled as replay final state rather than as the earlier Stage 2 baseline. Its document values are backed by `screenshots/replay/mongodb_after_replay.json` and the strict replay manifest.

![Mongo Express final document after Stage 3 replay](../screenshots/replay/mongodb_after_replay.png)


## Reflection

**Worked:** Spark committed offset 5 and MongoDB contained five unique metadata documents with the exact repository and dataset commit. **Failed:** that clean run had MongoDB query artifacts but no dedicated Mongo Express capture. **Resolution:** the canonical Stage 3 replay captured a real, hash-validated Mongo Express final-state image after replacement; it is presented above with its actual phase. Replay/idempotence remains Task 6, not a Stage 2 claim.
